In [ ]:
!pip install -q transformers sentence-transformers faiss-cpu accelerate sentencepiece torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 40.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

import faiss
import torch

from sentence_transformers import SentenceTransformer

from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModelForSeq2SeqLM
)

Upload Dataset

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving gst_hallucination_rag_dataset - gst_hallucination_rag_dataset.csv to gst_hallucination_rag_dataset - gst_hallucination_rag_dataset.csv


In [ ]:
df = pd.read_csv(next(iter(uploaded)))

df.head()

,query,answer,category
0,What is GST?,GST is a comprehensive indirect tax on goods a...,GST Basics
1,What is the objective of GST?,GST aims to create a unified tax system and co...,GST Basics
2,When was GST implemented in India?,GST was implemented on 1 July 2017.,GST Basics
3,What does GST stand for?,GST stands for Goods and Services Tax.,GST Basics
4,What is CGST?,CGST is the Central Goods and Services Tax lev...,GST Basics


Knowledge Base

In [ ]:
knowledge_base = []

for _, row in df.iterrows():

    text = f"""
Question:
{row['query']}

Answer:
{row['answer']}
"""

    knowledge_base.append(text)

print("Total Documents:", len(knowledge_base))

Total Documents: 20


Load Embedding Model

In [ ]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded


Generate Embeddings

In [ ]:
embeddings = embedding_model.encode(
    knowledge_base,
    convert_to_numpy=True,
    normalize_embeddings=True
)

FAISS Index

In [ ]:
index = faiss.IndexFlatIP(embeddings.shape[1])

index.add(embeddings)

Retriever Function

In [ ]:
def retrieve_context(query, top_k=3):

    # Normalize the query embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Search using cosine similarity (IndexFlatIP)
    scores, indices = index.search(
        query_embedding,
        top_k
    )

    contexts = []

    for idx in indices[0]:
        contexts.append(knowledge_base[idx])

    return "\n\n".join(contexts)

Test Retriever

In [ ]:
print(
    retrieve_context(
        "What is GST?"
    )
)


Question:
What is GST?

Answer:
GST is a comprehensive indirect tax on goods and services in India.



Question:
What does GST stand for?

Answer:
GST stands for Goods and Services Tax.



Question:
What is the objective of GST?

Answer:
GST aims to create a unified tax system and common national market.



Load Sensitive Information Detection Model

In [ ]:
from transformers import pipeline

ner_pipeline = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    tokenizer="dslim/bert-base-NER",
    aggregation_strategy="first"
)

print("Sensitive Information Detection Model Loaded Successfully")

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Sensitive Information Detection Model Loaded Successfully


Test the Model

In [ ]:
sample_text = """
My name is Rahul Sharma.
My email is rahul@gmail.com.
I live in Delhi.
"""

entities = ner_pipeline(sample_text)

entities

[{'entity_group': 'PER',
  'score': np.float32(0.99963623),
  'word': 'Rahul Sharma',
  'start': 12,
  'end': 24},
 {'entity_group': 'LOC',
  'score': np.float32(0.99940145),
  'word': 'Delhi',
  'start': 65,
  'end': 70}]

Create Sensitive Information Detector

In [ ]:
import re

def detect_sensitive_information(text):

    detected = []

    # -------------------------
    # Hugging Face NER
    # -------------------------

    entities = ner_pipeline(text)

    for entity in entities:

        detected.append({

            "Type": entity["entity_group"],

            "Value": entity["word"],

            "Score": round(entity["score"],4)

        })

    # Email

    emails = re.findall(
        r'[\w\.-]+@[\w\.-]+\.\w+',
        text
    )

    for email in emails:

        detected.append({

            "Type":"EMAIL",

            "Value":email,

            "Score":1.0

        })

    # Phone Number

    phones = re.findall(
        r'\b\d{10}\b',
        text
    )

    for phone in phones:

        detected.append({

            "Type":"PHONE",

            "Value":phone,

            "Score":1.0

        })

    # Aadhaar

    aadhaar = re.findall(
        r'\b\d{4}\s?\d{4}\s?\d{4}\b',
        text
    )

    for number in aadhaar:

        detected.append({

            "Type":"AADHAAR",

            "Value":number,

            "Score":1.0

        })

    # PAN

    pan = re.findall(
        r'\b[A-Z]{5}[0-9]{4}[A-Z]\b',
        text
    )

    for number in pan:

        detected.append({

            "Type":"PAN",

            "Value":number,

            "Score":1.0

        })

    return detected

Test the Detector

In [ ]:
query = """
Hi,

My name is Harshita Sharma.

My PAN number is ABCDE1234F.

My Aadhaar number is 1234 5678 9012.

My phone number is 9876543210.

My email is harshita@gmail.com.

Explain GST.
"""

results = detect_sensitive_information(query)

for item in results:

    print(item)

{'Type': 'PER', 'Value': 'Harshita Sharma', 'Score': np.float32(0.999)}
{'Type': 'ORG', 'Value': 'ABCDE1234F', 'Score': np.float32(0.3469)}
{'Type': 'ORG', 'Value': 'Aadhaar', 'Score': np.float32(0.7659)}
{'Type': 'EMAIL', 'Value': 'harshita@gmail.com', 'Score': 1.0}
{'Type': 'PHONE', 'Value': '9876543210', 'Score': 1.0}
{'Type': 'AADHAAR', 'Value': '1234 5678 9012', 'Score': 1.0}
{'Type': 'PAN', 'Value': 'ABCDE1234F', 'Score': 1.0}


Mask Sensitive Information

In [ ]:
def mask_sensitive_information(text):

    detected = detect_sensitive_information(text)

    masked_text = text

    priority = [
        "AADHAAR",
        "PAN",
        "EMAIL",
        "PHONE",
        "PER",
        "ORG",
        "LOC"
    ]

    detected = sorted(
        detected,
        key=lambda x: priority.index(x["Type"])
        if x["Type"] in priority else 999
    )

    seen = set()

    unique = []

    for item in detected:

        value = item["Value"]

        # Skip generic keywords

        if value.lower() in [
            "aadhaar",
            "pan",
            "email",
            "phone"
        ]:
            continue

        if value not in seen:

            unique.append(item)

            seen.add(value)

    for item in unique:

        masked_text = masked_text.replace(

            item["Value"],

            f"[{item['Type']}_MASKED]"

        )

    return masked_text

Test Masking

In [ ]:
masked = mask_sensitive_information(query)

print(masked)


Hi,

My name is [PER_MASKED].

My PAN number is [PAN_MASKED].

My Aadhaar number is [AADHAAR_MASKED].

My phone number is [PHONE_MASKED].

My email is [EMAIL_MASKED].

Explain GST.



Load FLAN-T5 Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

generator_tokenizer = AutoTokenizer.from_pretrained(
    "google/flan-t5-base"
)

generator_model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-base"
)

print("FLAN-T5 Loaded Successfully")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5 Loaded Successfully


Generate Function

In [ ]:
def generate_answer(query, context):

    prompt = f"""
Context:
{context}

Based on the above context, answer the following question.

Question: {query}

Answer:
"""

    inputs = generator_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = generator_model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False
    )

    return generator_tokenizer.decode(
        outputs[0], skip_special_tokens=True)

Test FLAN-T5

In [ ]:
query = "What is GST?"

context = retrieve_context(query)

answer = generate_answer(query, context)

print(answer)

GST is a comprehensive indirect tax on goods and services in India.


Security Report Function

In [ ]:
def security_report(query):

    detected = detect_sensitive_information(query)

    filtered = []
    seen = set()

    for item in detected:

        entity_type = item["Type"]
        value = item["Value"]

        # Ignore MISC
        if entity_type == "MISC":
            continue

        # Ignore generic words
        if value.lower() in ["aadhaar", "pan", "phone", "email"]:
            continue

        # Ignore ORG if it is actually a PAN number
        if entity_type == "ORG":

            if len(value) == 10 and value[:5].isalpha():
                continue

        key = (entity_type, value)

        if key not in seen:

            filtered.append({
                "Type": entity_type,
                "Value": value,
                "Confidence": round(float(item["Score"]),4)
            })

            seen.add(key)

    return {

        "Sensitive Information Found": len(filtered) > 0,

        "Total Entities": len(filtered),

        "Detected Entities": filtered

    }

Test Security Report

In [ ]:
query = """
My name is Harshita Sharma.

My PAN number is ABCDE1234F.

My Aadhaar number is 1234 5678 9012.

My email is harshita@gmail.com.

Explain GST.
"""

security_report(query)

{'Sensitive Information Found': True,
 'Total Entities': 4,
 'Detected Entities': [{'Type': 'PER',
   'Value': 'Harshita Sharma',
   'Confidence': 0.9992},
  {'Type': 'EMAIL', 'Value': 'harshita@gmail.com', 'Confidence': 1.0},
  {'Type': 'AADHAAR', 'Value': '1234 5678 9012', 'Confidence': 1.0},
  {'Type': 'PAN', 'Value': 'ABCDE1234F', 'Confidence': 1.0}]}

Secure RAG Pipeline

In [ ]:
def secure_rag(query):

    # Security report
    report = security_report(query)

    # Masked version
    masked_query = mask_sensitive_information(query)

    question = None

    for line in query.splitlines():
        line = line.strip()
        if line.endswith("?"):
            question = line

    if question is None:
        question = query.strip()

    context = retrieve_context(question, top_k=5)

    answer = generate_answer(question, context)

    return {
        "Original Query": query,
        "Masked Query": masked_query,
        "Question Used": question,
        "Security Report": report,
        "Retrieved Context": context,
        "Generated Answer": answer,
        "Processing Status": "Success"
    }

Display Results

In [ ]:
def display_result(result):

    print("="*90)

    print("ORIGINAL QUERY\n")
    print(result["Original Query"])

    print("\n"+"="*90)

    print("MASKED QUERY\n")
    print(result["Masked Query"])

    print("\n"+"="*90)

    print("SECURITY REPORT\n")

    report = result["Security Report"]

    print("Sensitive Information Found :", report["Sensitive Information Found"])
    print("Total Entities              :", report["Total Entities"])

    print("\nDetected Entities:\n")

    for entity in report["Detected Entities"]:

        print(entity)

    print("\n"+"="*90)

    print("RETRIEVED CONTEXT\n")
    print(result["Retrieved Context"])

    print("\n"+"="*90)

    print("GENERATED ANSWER\n")
    print(result["Generated Answer"])

    print("\n"+"="*90)

    print("PROCESSING STATUS\n")
    print(result["Processing Status"])

    print("="*90)

Test Result

In [ ]:
query = """
My name is Harshita Sharma.

My PAN number is ABCDE1234F.

My Aadhaar number is 1234 5678 9012.

My phone number is 9876543210.

My email is harshita@gmail.com.

What is GST?
"""

result = secure_rag(query)

display_result(result)

ORIGINAL QUERY


My name is Harshita Sharma.

My PAN number is ABCDE1234F.

My Aadhaar number is 1234 5678 9012.

My phone number is 9876543210.

My email is harshita@gmail.com.

What is GST?


MASKED QUERY


My name is [PER_MASKED].

My PAN number is [PAN_MASKED].

My Aadhaar number is [AADHAAR_MASKED].

My phone number is [PHONE_MASKED].

My email is [EMAIL_MASKED].

What is [MISC_MASKED]?


SECURITY REPORT

Sensitive Information Found : True
Total Entities              : 5

Detected Entities:

{'Type': 'PER', 'Value': 'Harshita Sharma', 'Confidence': 0.9992}
{'Type': 'EMAIL', 'Value': 'harshita@gmail.com', 'Confidence': 1.0}
{'Type': 'PHONE', 'Value': '9876543210', 'Confidence': 1.0}
{'Type': 'AADHAAR', 'Value': '1234 5678 9012', 'Confidence': 1.0}
{'Type': 'PAN', 'Value': 'ABCDE1234F', 'Confidence': 1.0}

RETRIEVED CONTEXT


Question:
What is GST?

Answer:
GST is a comprehensive indirect tax on goods and services in India.



Question:
What does GST stand for?

Answer:
GST stands f